# Chapter 10 &mdash; Rule 6: The Derivative of a Negation, and Why Deferring Works

**Concept 5 of the Chapter 10 decomposition:** *Rule 6: The Derivative of a Negation, and Why Deferring Works*

$(!E)_c = !(E_c)$ &mdash; differentiate inside, and keep the negation waiting for the nullability test.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter10/Concept-Rule-6-Negation/Concept-Rule-6-Negation.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_rederiv    import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


$$(!E)_c = !(E_c)$$

Differentiate **inside** the negation and leave the `!` in place. The rule looks too
easy, and the reason it works is worth stating:

$$w \in L(!E) \iff w \notin L(E) \iff \text{(inductively)}\ w' \notin L(E_c) \iff w' \in L(!(E_c)).$$

Complementation **commutes with taking derivatives**, because both are defined
pointwise on strings.

All the real work is **deferred** to `nullable`, where negation is a single `not`. The
classical route has to determinize *before* it can complement; here nothing is
determinized at all.

## 2. Definitions

### The matcher

In [ ]:
# --- the derivative matcher, in full -------------------------------------
# AST forms produced by re2ast:
#    ('@','@')            epsilon
#    ('str', c)           a single symbol
#    ('+', (E1, E2))      union
#    ('.', (E1, E2))      concatenation
#    ('*', E)             star
#    ('!', E)             negation
#    ('&', (E1, E2))      intersection
EPS   = ('@', '@')
PHI   = ('phi', 'phi')          # the empty language -- not produced by the
                                # parser, but the derivative rules need it

def nullable(E):
    t = E[0]
    if t == '@'  : return True
    if t == 'phi': return False
    if t == 'str': return False
    if t == '+'  : return nullable(E[1][0]) or  nullable(E[1][1])
    if t == '&'  : return nullable(E[1][0]) and nullable(E[1][1])
    if t == '.'  : return nullable(E[1][0]) and nullable(E[1][1])
    if t == '*'  : return True
    if t == '!'  : return not nullable(E[1])
    raise ValueError(E)

def dv(c, E):
    t = E[0]
    if t == '@'  : return PHI
    if t == 'phi': return PHI
    if t == 'str': return EPS if E[1] == c else PHI
    if t == '+'  : return ('+', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '&'  : return ('&', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '*'  : return ('.', (dv(c, E[1]), E))
    if t == '!'  : return ('!', dv(c, E[1]))
    if t == '.'  :
        E1, E2 = E[1]
        left = ('.', (dv(c, E1), E2))
        return ('+', (left, dv(c, E2))) if nullable(E1) else left
    raise ValueError(E)

def matches(s, E):
    for ch in s:
        E = dv(ch, E)
    return nullable(E)

def rmatch(restr, s):
    return matches(s, re2ast(restr)[0])

### A reference complement, built the classical way

In [ ]:
def re_dfa(r): return min_dfa(nfa2dfa(re2nfa(r)))
from itertools import product
STRS = [''.join(p) for k in range(10) for p in product('01', repeat=k)]

## 3. Tests

The rule, applied once.

In [ ]:
E  = re2ast("!(0*)")[0]
print("E        =", E)
print("dv('1',E) =", dv('1', E))
assert E[0] == '!' and dv('1', E)[0] == '!'
print("\nthe '!' is still on the outside -- only its argument changed")

$!E$ matches exactly the complement.

In [ ]:
# every pattern here mentions BOTH symbols, so its DFA's Sigma is {0,1}
# and the shared STRS list can be used directly
for restr in ["(0+1)*01", "(01)*", "0*1*", "1(0+1)*0"]:
    E = re2ast("!(%s)" % restr)[0]
    D = comp_dfa(re_dfa(restr))
    bad = [s for s in STRS if matches(s, E) != accepts_dfa(D, s)]
    print("!(%-12s) vs comp_dfa : %d mismatches" % (restr, len(bad)))
    assert not bad

Double negation is the identity, as it must be.

In [ ]:
E1 = re2ast("(0+1)*01")[0]
E2 = re2ast("!(!((0+1)*01))")[0]
assert all(matches(s, E1) == matches(s, E2) for s in STRS)
print("!!E and E agree on all %d strings" % len(STRS))

**Where the work went:** `nullable` is the only place negation is resolved.

In [ ]:
E = re2ast("!(0*)")[0]
print("nullable(!(0*)) =", nullable(E), "  (0* IS nullable, so its negation is not)")
assert not nullable(E)
E = re2ast("!(00*)")[0]
print("nullable(!(00*)) =", nullable(E), "  (00* is not nullable, so its negation is)")
assert nullable(E)
print("\nOne 'not'.  That is the whole cost of complementation.")

Compare the two routes on the same language.

In [ ]:
r = "(0+1)*0101(0+1)*"
print("classical : re2nfa -> nfa2dfa -> totalize -> flip F")
print("            intermediate DFA has %d states" % len(nfa2dfa(re2nfa(r))["Q"]))
print("derivative: one line in dv(), one 'not' in nullable()")
E = re2ast("!(%s)" % r)[0]
D = comp_dfa(re_dfa(r))
assert all(matches(s, E) == accepts_dfa(D, s) for s in STRS)
print("and they agree on all %d strings" % len(STRS))

## 4. Exercises


1. Why does complementation commute with derivatives but **not** with, say, reversal?
2. What is $(!\emptyset)_c$? What language is $!\emptyset$?
3. Express $E_1 - E_2$ with `!` and `&`, then check your rule.

In [ ]:
# Your work for the exercises above.